# 01 — AI Hub 감귤 데이터셋 탐색적 분석 (EDA)

이 노트북은 **AI Hub 감귤 궤양병 데이터셋**의 구성을 이해하기 위한 탐색적 데이터 분석(EDA)입니다.
모델 학습 전에 데이터 품질·분포·편향을 파악하는 것이 목적입니다.

## What you'll learn
- 클래스(정상 / 궤양병) 별 이미지 수와 분포
- Polygon 라벨 커버리지 (전체 대비 polygon 존재 비율)
- 카메라·촬영지·생육단계 등 메타데이터 분포
- 기상 환경 데이터(기온, 습도, 일사량 등) 분포
- 촬영 시기(월별) 분포


## Setup

In [ ]:
import os, sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# project root → sys.path (노트북 실행 위치가 project root 라고 가정)
from pathlib import Path
PROJECT_ROOT = Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2

from common.label_parser import load_sample, polygon_to_mask

# matplotlib 한글 폰트 설정 (macOS)
import matplotlib.font_manager as fm
_korean_candidates = [
    "/System/Library/Fonts/Supplemental/AppleGothic.ttf",
    "/Library/Fonts/NanumGothic.ttf",
]
for _fc in _korean_candidates:
    if Path(_fc).exists():
        fm.fontManager.addfont(_fc)
        matplotlib.rcParams["font.family"] = fm.FontProperties(fname=_fc).get_name()
        break
matplotlib.rcParams["axes.unicode_minus"] = False

DATABASE_ROOT = PROJECT_ROOT / "database"
print("Project root :", PROJECT_ROOT)
print("Database root:", DATABASE_ROOT)
print("Database exists:", DATABASE_ROOT.exists())


## 1. 디렉토리 구조 및 파일 수 집계

`database/` 하위 디렉토리를 순회하여 split × class 별 이미지/JSON/polygon 수를 집계합니다.


In [ ]:
SPLIT_DIRS = {
    "train": {
        "split": "1.Training",
        "img": "원천데이터/TS1.감귤",
        "lbl": "라벨링데이터/TL1.감귤",
    },
    "val": {
        "split": "2.Validation",
        "img": "원천데이터/VS1.감귤",
        "lbl": "라벨링데이터/VL1.감귤",
    },
}
CLASS_DIRS = ["열매_정상", "열매_궤양병"]

records = []
for split_name, cfg in SPLIT_DIRS.items():
    split_dir = DATABASE_ROOT / cfg["split"]
    for cls_dir in CLASS_DIRS:
        img_dir = split_dir / cfg["img"] / cls_dir
        lbl_dir = split_dir / cfg["lbl"] / cls_dir
        if not lbl_dir.exists():
            print(f"  [SKIP] {lbl_dir} not found")
            continue
        json_files = sorted(lbl_dir.glob("*.json"))
        img_files  = sorted(img_dir.glob("*")) if img_dir.exists() else []
        img_files  = [f for f in img_files if f.suffix.lower() in (".jpg", ".jpeg", ".png")]
        polygon_count = sum(1 for jp in json_files if load_sample(jp)["has_polygon"])
        records.append({
            "split": split_name,
            "class": cls_dir,
            "image_count": len(img_files),
            "json_count":  len(json_files),
            "polygon_count": polygon_count,
        })

df_counts = pd.DataFrame(records)
df_counts["polygon_ratio"] = (df_counts["polygon_count"] / df_counts["json_count"]).round(4)
print(df_counts.to_string(index=False))


## 2. 클래스 분포 시각화

train / val 각각의 클래스 수와 polygon 커버리지를 비교합니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- (A) 이미지 수 ---
ax = axes[0]
for i, (split_name, grp) in enumerate(df_counts.groupby("split")):
    x_pos = [j + i * 0.35 for j in range(len(grp))]
    bars = ax.bar(x_pos, grp["image_count"], width=0.35, label=split_name)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(int(bar.get_height())), ha="center", va="bottom", fontsize=9)
classes = [c.replace("열매_", "") for c in df_counts["class"].unique()]
ax.set_xticks([j + 0.175 for j in range(len(classes))])
ax.set_xticklabels(classes)
ax.set_title("이미지 수 (split × class)")
ax.set_ylabel("이미지 수")
ax.legend()

# --- (B) Polygon 커버리지 ---
ax2 = axes[1]
for i, (split_name, grp) in enumerate(df_counts.groupby("split")):
    x_pos = [j + i * 0.35 for j in range(len(grp))]
    bars = ax2.bar(x_pos, grp["polygon_ratio"] * 100, width=0.35, label=split_name)
    for bar in bars:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=9)
ax2.set_xticks([j + 0.175 for j in range(len(classes))])
ax2.set_xticklabels(classes)
ax2.set_title("Polygon 커버리지 (%) — JSON 대비")
ax2.set_ylabel("비율 (%)")
ax2.set_ylim(0, 110)
ax2.legend()

plt.tight_layout()
plt.show()


## 3. 이미지 샘플 및 Polygon Overlay

정상(열매_정상) / 궤양병(열매_궤양병) 각 1장씩, polygon이 있는 궤양병 샘플에는 마스크를 overlay합니다.


In [ ]:
def pick_sample(split, cls_dir, want_polygon=False):
    """Return (img_path, json_path) for one sample matching criteria."""    cfg = SPLIT_DIRS[split]
    lbl_dir = DATABASE_ROOT / cfg["split"] / cfg["lbl"] / cls_dir
    img_dir = DATABASE_ROOT / cfg["split"] / cfg["img"] / cls_dir
    for jp in sorted(lbl_dir.glob("*.json")):
        info = load_sample(jp)
        if want_polygon and not info["has_polygon"]:
            continue
        cands = list(img_dir.glob(f"{jp.stem}.*"))
        cands = [c for c in cands if c.suffix.lower() in (".jpg", ".jpeg", ".png")]
        if cands:
            return cands[0], jp
    return None, None

samples = [
    ("train", "열매_정상",   False, "Train — 정상 (no polygon)"),
    ("train", "열매_궤양병", False, "Train — 궤양병 (no overlay)"),
    ("train", "열매_궤양병", True,  "Train — 궤양병 + polygon overlay"),
    ("val",   "열매_정상",   False, "Val — 정상"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (split, cls, want_poly, title) in zip(axes.flatten(), samples):
    img_path, jp = pick_sample(split, cls, want_polygon=want_poly)
    if img_path is None:
        ax.set_title(f"{title}\n(샘플 없음)")
        ax.axis("off")
        continue
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    if want_poly:
        info = load_sample(jp)
        if info["has_polygon"]:
            h, w = img_rgb.shape[:2]
            mask = polygon_to_mask(info["polygon"], h, w)
            # red overlay (alpha blend)
            overlay = img_rgb.copy()
            overlay[mask == 1] = (overlay[mask == 1] * 0.5 + np.array([200, 30, 30]) * 0.5).astype(np.uint8)
            img_rgb = overlay
    ax.imshow(img_rgb)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

plt.suptitle("이미지 샘플 (2×2 그리드)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 4. 메타데이터 EDA

모든 JSON 라벨에서 메타데이터를 수집하여 DataFrame으로 만든 뒤 분포를 시각화합니다.
(전체 집계이므로 처음 실행 시 수 분 소요될 수 있습니다.)


In [ ]:
all_records = []
for split_name, cfg in SPLIT_DIRS.items():
    split_dir = DATABASE_ROOT / cfg["split"]
    for cls_dir in CLASS_DIRS:
        lbl_dir = split_dir / cfg["lbl"] / cls_dir
        if not lbl_dir.exists():
            continue
        for jp in sorted(lbl_dir.glob("*.json")):
            info = load_sample(jp)
            m = info["metadata"]
            e = m["env"]
            all_records.append({
                "split":        split_name,
                "class":        cls_dir,
                "has_polygon":  info["has_polygon"],
                "camera":       m.get("camera"),
                "location":     m.get("location"),
                "place_type":   m.get("place_type"),
                "growth_stage": m.get("growth_stage"),
                "date":         m.get("date"),
                "solar":        e["solar"],
                "rain":         e["rain"],
                "temp":         e["temp"],
                "humidity":     e["humidity"],
                "soil_moisture":e["soil_moisture"],
            })

df_meta = pd.DataFrame(all_records)
print(f"총 샘플 수: {len(df_meta):,}")
print(df_meta.dtypes)
df_meta.head(3)


In [ ]:
# 범주형 메타데이터 분포 — 4개 subplot
cat_cols = ["camera", "location", "place_type", "growth_stage"]
cat_labels = ["카메라", "촬영지(location)", "촬영 장소 유형", "생육단계"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, col, lbl in zip(axes.flatten(), cat_cols, cat_labels):
    vc = df_meta[col].value_counts(dropna=False).head(20)
    vc.plot(kind="bar", ax=ax)
    ax.set_title(lbl + " 분포")
    ax.set_xlabel("")
    ax.set_ylabel("샘플 수")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# 환경 수치 데이터 히스토그램
env_cols   = ["temp", "humidity", "solar", "soil_moisture", "rain"]
env_labels = ["기온 (°C)", "상대습도 (%)", "일사량 (MJ/m²)", "토양수분 (%)", "강수량 (mm)"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col, lbl in zip(axes.flatten(), env_cols, env_labels):
    vals = df_meta[col].replace(0, np.nan).dropna()
    if len(vals) == 0:
        ax.set_title(lbl + "\n(데이터 없음)")
        ax.axis("off")
        continue
    ax.hist(vals, bins=40, edgecolor="white", linewidth=0.5)
    ax.set_title(lbl)
    ax.set_ylabel("빈도")

# 마지막 subplot 숨기기
axes[-1, -1].axis("off")
plt.suptitle("환경 수치 데이터 분포", fontsize=13)
plt.tight_layout()
plt.show()


## 5. 촬영 시기(월) 분포

`OCPRD` 필드에서 월(month)을 추출해 계절적 패턴을 확인합니다.


In [ ]:
import re

def extract_month(date_str):
    if not date_str:
        return None
    # 예시 형식: '20230815', '2023-08-15', '2023.08.15'
    m = re.search(r"(\d{4})[-.]?(\d{2})[-.]?(\d{2})", str(date_str))
    if m:
        return int(m.group(2))
    return None

df_meta["month"] = df_meta["date"].apply(extract_month)

month_counts = df_meta.groupby(["month", "class"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))
month_counts.plot(kind="bar", ax=ax)
ax.set_title("촬영 월별 샘플 분포")
ax.set_xlabel("월 (month)")
ax.set_ylabel("샘플 수")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

print("월별 총계:\n", df_meta["month"].value_counts().sort_index())


## 6. 관찰 요약

데이터 탐색 결과에서 주목할 점을 정리합니다.

- **클래스 불균형**: 정상(열매_정상) 이미지가 궤양병(열매_궤양병) 이미지보다 많은 경향이 있습니다.
  학습 시 Weighted Sampler 또는 클래스 가중치를 적용할 필요가 있습니다.
- **Polygon 커버리지 편중**: 전체 JSON 라벨 중 polygon이 있는 비율은 궤양병 클래스에 집중되어 있습니다.
  정상 클래스는 polygon 없이 이미지 레벨 라벨만 존재하는 경우가 많습니다.
- **카메라·촬영 장소 편향**: 특정 카메라 모델 또는 촬영 지역에 샘플이 집중될 경우 도메인 쉬프트가 발생할 수 있습니다.
- **기상 데이터 결측**: 일부 환경 수치(일사량, 토양수분 등)에 0 또는 결측값이 많아 직접 활용 전 전처리가 필요합니다.


## 📝 Your turn

아래 질문에 답하는 분석을 직접 추가해보세요.

1. **Polygon 라벨이 일부에만 존재하는 이유**는 무엇일까요?
   AI Hub 라벨링 가이드라인이나 annotation 스키마를 찾아보고, 궤양병 클래스에만 polygon이 있는 이유를 설명해보세요.

2. **카메라별 불균형이 도메인 쉬프트 문제로 이어질까요?**
   특정 카메라로 찍힌 이미지가 train/val에 어떻게 분포하는지 확인하고, 쉬프트 가능성을 평가해보세요.

3. **기상 데이터와 궤양병 발생 사이에 상관관계가 있을까요?**
   `df_meta`에서 class=열매_궤양병 vs 열매_정상 그룹을 나눠 환경 변수(기온, 습도)의 분포를 비교·시각화해보세요.

4. **생육단계별 궤양병 발생률**은 어떻게 다를까요?
   `growth_stage × class` 교차 테이블을 만들어 분석해보세요.

5. **데이터 품질 이슈**가 있나요?
   이미지 파일은 있는데 JSON이 없는 케이스, 또는 반대 케이스가 있는지 확인하고 비율을 구해보세요.
